# Midfielder Interceptions Model

Predict **interceptions per 90 minutes** for midfielders.

- Train: GW 20-32 | Test: GW 33-35 | Predict: GW 36+
- Filter: minutes_played > 30 (avoid sub outliers)
- Scoring: +2 pts per interception (MID only)
- At prediction time, scale by assumed minutes: `int_pts = pred_int_per_90 × (mins/90) × 2`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
from pathlib import Path

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_style('whitegrid')
print('Imports OK')

## 1. Load Data

In [ ]:
data_dir = Path('../../data')
csv_files = sorted(data_dir.glob('player_stats_gw*.csv'))

dfs = [pd.read_csv(f) for f in csv_files]
df_all = pd.concat(dfs, ignore_index=True)

# Filter: midfielders only, played > 30 mins (avoid sub outliers)
df = df_all[(df_all['position'] == 'MID') & (df_all['minutes_played'] > 30)].copy()
df = df.sort_values(['player_id', 'gameweek']).reset_index(drop=True)

print(f'Loaded {len(csv_files)} gameweeks')
print(f'All rows: {len(df_all):,}')
print(f'MID rows (mins > 30): {len(df):,}')
print(f'Gameweeks: {df["gameweek"].min()}-{df["gameweek"].max()}')

## 2. Load Fixtures & Squads

Reads from pre-fetched data files. Run `python data/fetch_fixtures.py` to update.

In [ ]:
# Load pre-fetched fixture and squad data
with open(data_dir / 'fixtures_remaining.json') as f:
    remaining_fixtures = json.load(f)

with open(data_dir / 'squads.json') as f:
    squads_data = json.load(f)

squads_by_id = {s['id']: s for s in squads_data}

print(f'Loaded {len(remaining_fixtures)} future gameweeks')
print(f'Loaded {len(squads_data)} squads')
for gw, games in sorted(remaining_fixtures.items(), key=lambda x: int(x[0])):
    print(f'  GW {gw}: {len(games)} games')

## 3. Build Features

In [ ]:
# Target: interceptions per 90 minutes
df['int_per_90'] = df['interceptions'] / (df['minutes_played'] / 90)

# Player features (per-player via transform)
df['int_per_90_last10'] = df.groupby('player_id')['int_per_90'].transform(
    lambda x: x.shift(1).rolling(window=10, min_periods=1).mean()
)
df['int_per_90_season'] = df.groupby('player_id')['int_per_90'].transform(
    lambda x: x.shift(1).expanding(min_periods=1).mean()
)

# Opposition feature: avg interceptions MIDs make against this opponent
opp_stats = df.groupby('opponent_id')['interceptions'].mean().reset_index()
opp_stats.columns = ['opponent_id', 'opp_avg_int_conceded']
df = df.merge(opp_stats, on='opponent_id', how='left')
df['opp_avg_int_conceded'] = df['opp_avg_int_conceded'].fillna(df['interceptions'].mean())

# Match context
difficulty_map = {'easy': 0, 'medium': 1, 'hard': 2}
df['fixture_diff'] = df['fixture_difficulty'].map(difficulty_map).fillna(1)
df['is_home'] = (df['is_home'] == 'H').astype(int)

print('Features built.')
print(f'\nInt/90 stats:')
print(df['int_per_90'].describe().round(3))

## 4. Train & Test Split

In [ ]:
# Filter to GW >= 20, drop rows without enough history
df_model = df[df['gameweek'] >= 20].copy()
df_model = df_model.dropna(subset=['int_per_90_last10', 'int_per_90_season'])

features = ['int_per_90_last10', 'int_per_90_season', 'opp_avg_int_conceded', 'fixture_diff', 'is_home']
target = 'int_per_90'

print(f'Model rows (GW >= 20): {len(df_model):,}')
print(f'Train (GW 20-32): {len(df_model[df_model["gameweek"] <= 32]):,}')
print(f'Test  (GW 33-35): {len(df_model[df_model["gameweek"].isin([33,34,35])]):,}')

## 5. Walk-Forward Validation (GW 33-35)

In [ ]:
test_gws = [33, 34, 35]
results = []

for test_gw in test_gws:
    train = df_model[df_model['gameweek'] < test_gw]
    test = df_model[df_model['gameweek'] == test_gw]
    
    if len(train) == 0 or len(test) == 0:
        print(f'GW {test_gw}: Skipped')
        continue
    
    model = LinearRegression().fit(train[features], train[target])
    y_pred = model.predict(test[features])
    y_test = test[target].values
    
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    
    results.append({'GW': test_gw, 'MAE': mae, 'RMSE': rmse, 'R2': r2})
    print(f'GW {test_gw}: MAE={mae:.3f}, RMSE={rmse:.3f}, R²={r2:.3f} (n={len(test)})')

print(f'\n=== Average Metrics ===')
print(f'MAE:  {np.mean([r["MAE"] for r in results]):.3f}')
print(f'RMSE: {np.mean([r["RMSE"] for r in results]):.3f}')
print(f'R²:   {np.mean([r["R2"] for r in results]):.3f}')

# Feature coefficients
print(f'\n=== Feature Coefficients ===')
for feat, coef in sorted(zip(features, model.coef_), key=lambda x: abs(x[1]), reverse=True):
    print(f'  {feat:25s} {coef:+.4f}')
print(f'  {"intercept":25s} {model.intercept_:+.4f}')

## 6. Case Study

In [ ]:
def int_predictions(player_name, assumed_mins=90):
    """Predict interceptions for a midfielder (GW 33-35 test period).
    
    Predicts int/90, then scales: int_pts = pred_int_per_90 × (assumed_mins/90) × 2
    """
    player_data = df_model[df_model['display_name'].str.contains(player_name, case=False, na=False)]
    if len(player_data) == 0:
        print(f'Player "{player_name}" not found')
        return
    
    player_id = player_data['player_id'].iloc[0]
    full_name = player_data['display_name'].iloc[0]
    
    player_test = df_model[(df_model['player_id'] == player_id) & 
                           (df_model['gameweek'].isin([33, 34, 35]))]
    if len(player_test) == 0:
        print(f'No test data for {full_name}')
        return
    
    # Train on GW < 33
    train = df_model[df_model['gameweek'] < 33]
    model = LinearRegression().fit(train[features], train[target])
    
    y_pred = model.predict(player_test[features])
    y_actual = player_test['int_per_90'].values
    mins_scale = assumed_mins / 90
    
    print(f'=== {full_name}: Interceptions (GW 33-35) ===')
    print(f'    Assumed minutes: {assumed_mins}\n')
    
    rows = []
    for idx, (_, row) in enumerate(player_test.iterrows()):
        actual_mins = int(row['minutes_played'])
        actual_scale = actual_mins / 90
        rows.append({
            'GW': int(row['gameweek']),
            'Opponent': row['opponent_name'],
            'H/A': 'H' if row['is_home'] == 1 else 'A',
            'Int/90 Szn': f"{row['int_per_90_season']:.2f}",
            'Int/90 L10': f"{row['int_per_90_last10']:.2f}",
            'Opp Conc': f"{row['opp_avg_int_conceded']:.2f}",
            'Mins': actual_mins,
            'Pred Int/90': f"{y_pred[idx]:.2f}",
            'Act Int/90': f"{y_actual[idx]:.2f}",
            f'Pts ({assumed_mins}m)': f"{y_pred[idx] * mins_scale * 2:.1f}",
            f'Pts (act m)': f"{y_pred[idx] * actual_scale * 2:.1f}",
            'Pts actual': f"{row['interceptions'] * 2}",
        })
    
    print(pd.DataFrame(rows).to_string(index=False))
    
    mae = mean_absolute_error(y_actual, y_pred)
    rmse = np.sqrt(mean_squared_error(y_actual, y_pred))
    print(f'\nInt/90 — MAE: {mae:.2f}, RMSE: {rmse:.2f}')

int_predictions('L. Wing')
print()
int_predictions('S. Bradley')

## 7. GW 36+ Predictions

In [ ]:
# Train final model on ALL available data (GW 20-35)
model_final = LinearRegression().fit(df_model[features], df_model[target])

# Load fixtures
with open(data_dir / 'fixtures_remaining.json') as f:
    remaining_fixtures = json.load(f)

# Get latest player features (most recent row per player)
latest = df.sort_values('gameweek').groupby('player_id').last().reset_index()

# Build predictions for GW 36
gw = '36'
if gw not in remaining_fixtures:
    print(f'No fixture data for GW {gw}')
else:
    fixtures_gw36 = remaining_fixtures[gw]
    
    # Build squad-to-fixtures lookup
    squad_fixtures = {}
    for fix in fixtures_gw36:
        # Home team
        hid = fix['home_id']
        if hid not in squad_fixtures:
            squad_fixtures[hid] = []
        squad_fixtures[hid].append({
            'opponent_name': fix['away_name'],
            'opponent_id': fix['away_id'],
            'is_home': 1,
            'fixture_diff': {'easy': 0, 'medium': 1, 'hard': 2}.get(fix['home_difficulty'], 1)
        })
        # Away team
        aid = fix['away_id']
        if aid not in squad_fixtures:
            squad_fixtures[aid] = []
        squad_fixtures[aid].append({
            'opponent_name': fix['home_name'],
            'opponent_id': fix['home_id'],
            'is_home': 0,
            'fixture_diff': {'easy': 0, 'medium': 1, 'hard': 2}.get(fix['away_difficulty'], 1)
        })
    
    # Predict for each MID
    predictions = []
    for _, player in latest.iterrows():
        if pd.isna(player.get('int_per_90_last10')) or pd.isna(player.get('int_per_90_season')):
            continue
        
        squad_id = player['squad_id']
        if squad_id not in squad_fixtures:
            continue
        
        for fix in squad_fixtures[squad_id]:
            # Look up opp_avg_int_conceded for this opponent
            opp_conc = opp_stats[opp_stats['opponent_id'] == fix['opponent_id']]['opp_avg_int_conceded']
            opp_val = opp_conc.values[0] if len(opp_conc) > 0 else df['interceptions'].mean()
            
            X = pd.DataFrame([{
                'int_per_90_last10': player['int_per_90_last10'],
                'int_per_90_season': player['int_per_90_season'],
                'opp_avg_int_conceded': opp_val,
                'fixture_diff': fix['fixture_diff'],
                'is_home': fix['is_home'],
            }])
            
            pred_int_per_90 = model_final.predict(X)[0]
            pred_pts_90m = pred_int_per_90 * 2  # assuming 90 mins
            
            predictions.append({
                'Player': player['display_name'],
                'Team': squads_by_id.get(squad_id, {}).get('shortName', ''),
                'Opponent': fix['opponent_name'],
                'H/A': 'H' if fix['is_home'] else 'A',
                'Difficulty': ['easy', 'medium', 'hard'][fix['fixture_diff']],
                'Int/90 Szn': f"{player['int_per_90_season']:.2f}",
                'Int/90 L10': f"{player['int_per_90_last10']:.2f}",
                'Pred Int/90': round(pred_int_per_90, 2),
                'Pred Pts (90m)': round(pred_pts_90m, 1),
            })
    
    pred_df = pd.DataFrame(predictions).sort_values('Pred Pts (90m)', ascending=False)
    print(f'=== GW {gw} Interception Predictions (Top 25, assuming 90 mins) ===')
    print(pred_df.head(25).to_string(index=False))
    print(f'\nTotal MID predictions: {len(pred_df)}')